## Question 1

In [1]:
from pathlib import Path
from statistics import mean
import requests
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework05') \
    .getOrCreate()

25/03/01 14:44:54 WARN Utils: Your hostname, dm-Latitude-5401 resolves to a loopback address: 127.0.1.1; using 192.168.8.101 instead (on interface wlo1)
25/03/01 14:44:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/03/01 14:44:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print(f"Answer 1: {spark.version=}")

Answer 1: spark.version='3.3.2'


## Question 2

In [4]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet"
fname = Path(url.split('/')[-1])

if not Path.is_file(fname):
    response = requests.get(url)

    with open(fname, 'wb') as fd:
        fd.write(response.content)

In [5]:
df = spark.read.parquet(fname.as_posix())

In [6]:
output_path = Path("data/pq/")

df \
    .repartition(4) \
    .write \
    .parquet(output_path.as_posix(), mode='overwrite')

In [7]:
output_pq_fpaths = [
    fpath
    for fpath in Path.iterdir(output_path)
    if fpath.suffix == '.parquet'
]

pq_file_sizes_mb = [
    fpath.stat().st_size / 2**20
    for fpath in output_pq_fpaths
]

avg_pq_file_size = mean(pq_file_sizes_mb)
print(f"Answer 2: average parquet file size = {round(avg_pq_file_size, 1)}")

Answer 2: average parquet file size = 24.2


## Question 3

In [8]:
the_date = '2024-10-15'

num_trips = df \
    .withColumn('pickup_date', F.to_date(df.tpep_pickup_datetime)) \
    .filter(f"pickup_date = '{the_date}'") \
    .count()

print(f"Answer 3: number of trips on {the_date} was = {num_trips}")

Answer 3: number of trips on 2024-10-15 was = 128097


## Question 4

In [9]:
max_duration_seconds = df \
    .withColumn(
        'duration',
        df.tpep_dropoff_datetime.cast('long') - df.tpep_pickup_datetime.cast('long')
    ) \
    .select(F.max('duration')) \
    .collect()[0][0]

print(f"Answer 4: longest trip duration (in hours) = {round(max_duration_seconds / (60 * 60), 2)}")

Answer 4: longest trip duration (in hours) = 162.62


## Question 5
The Spark Jobs dashboard is at localhost:4040.

## Question 6

In [10]:
url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
fname = Path(url.split('/')[-1])

if not Path.is_file(fname):
    response = requests.get(url)

    with open(fname, 'wb') as fd:
        fd.write(response.content)

In [11]:
df_zones = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv(fname.as_posix())

In [12]:
df_zones.createOrReplaceTempView("zones_IDs_and_names")

In [13]:
least_frequent_pickup_location_id = df \
    .groupBy('PULocationID') \
    .count() \
    .orderBy('count') \
    .select('PULocationID') \
    .first()[0]

least_frequent_pickup_location_id

105

In [14]:
least_frequent_pickup_zone_name = df_zones \
    .filter(f"LocationID = {least_frequent_pickup_location_id}") \
    .select('Zone') \
    .first()[0]

print(f"Answer 6: least frequent pickup location Zone = {least_frequent_pickup_zone_name}")

Answer 6: least frequent pickup location Zone = Governor's Island/Ellis Island/Liberty Island
